# Week 04 — Cleaning a CSV for a Spatial Join

In [14]:
import csv
import unicodedata
from collections import Counter
from pathlib import Path

RAW_PATH = Path("data/us_county_fips_raw.csv")
CLEAN_PATH = Path("data/us_county_fips_clean.csv")

In [15]:
with RAW_PATH.open(newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)         
    raw_rows = list(reader)

print(reader.fieldnames)
print(len(raw_rows))

# FIPS string-length distribution
len_dist = Counter(len(r["fips"].strip()) for r in raw_rows)

aggregates = [r for r in raw_rows
              if r["fips"].strip().zfill(5).endswith("000") or r["state"].strip() == "NA"]

bad = next(r for r in raw_rows if r["fips"].strip() == "35013")

FileNotFoundError: [Errno 2] No such file or directory: 'data/us_county_fips_raw.csv'

## 2. Cleaning plan
| Step | Problem | Fix |
|------|---------|-----|
| a | Leading zeros dropped | `fips.zfill(5)` — restore the 5-digit string key |
| b | Stray whitespace | `.strip()` every field |
| c | Broken encoding | targeted repair keyed on FIPS + drop orphaned combining marks, then NFC-normalize |
| d | `NA` / non-county rows | drop nation + state aggregates (no polygon to join to) |
| e | Bad records | validate FIPS is 5 digits, `state` is a real 2-letter code, `name` non-empty |
| f | Duplicates | keep first occurrence per FIPS |

Output: `data/us_county_fips_clean.csv` with a stable schema `fips, name, state`, one row per US county.

In [ ]:
# --- helpers -------------------------------------------------------------
# Known encoding artifacts, keyed by FIPS so the correction is unambiguous.
NAME_FIXES = {"35013": "Do\u00f1a Ana County"}  # -> Doña Ana County, NM

VALID_STATE_ABBR = {
    "AL","AK","AZ","AR","CA","CO","CT","DE","DC","FL","GA","HI","ID","IL","IN",
    "IA","KS","KY","LA","ME","MD","MA","MI","MN","MS","MO","MT","NE","NV","NH",
    "NJ","NM","NY","NC","ND","OH","OK","OR","PA","RI","SC","SD","TN","TX","UT",
    "VT","VA","WA","WV","WI","WY",
}

def strip_combining(text):
    """Remove orphaned combining marks left behind by a bad encoding."""
    return "".join(c for c in text if not unicodedata.combining(c))

In [ ]:
# --- the cleaning pass ---------------------------------------------------
clean_rows = []
seen = set()
stats = Counter()

for row in raw_rows:
    fips  = (row.get("fips")  or "").strip()   # (b) strip whitespace
    name  = (row.get("name")  or "").strip()
    state = (row.get("state") or "").strip()

    # (a) restore 5-digit zero-padded FIPS — the join-breaking bug
    if fips.isdigit():
        padded = fips.zfill(5)
        if padded != fips:
            stats["fips_padded"] += 1
        fips = padded

    # (d) drop nation/state aggregate rows — nothing to join them to
    if fips.endswith("000") or state.upper() == "NA":
        stats["dropped_aggregate"] += 1
        continue

    # (c) repair encoding, then normalize
    if fips in NAME_FIXES:
        name = NAME_FIXES[fips]
        stats["name_repaired"] += 1
    name  = unicodedata.normalize("NFC", strip_combining(name))
    state = state.upper()

    # (e) validate
    if not (len(fips) == 5 and fips.isdigit()) or state not in VALID_STATE_ABBR or not name:
        stats["dropped_invalid"] += 1
        continue

    # (f) dedupe on the join key
    if fips in seen:
        stats["duplicates"] += 1
        continue
    seen.add(fips)
    clean_rows.append({"fips": fips, "name": name, "state": state})

clean_rows.sort(key=lambda r: r["fips"])   # stable, join-friendly ordering

for k in ("fips_padded","name_repaired","dropped_aggregate","dropped_invalid","duplicates"):
    print(f"{k:18}: {stats[k]}")
print(f"{'clean rows':18}: {len(clean_rows)}")

fips_padded       : 324
name_repaired     : 1
dropped_aggregate : 52
dropped_invalid   : 0
duplicates        : 0
clean rows        : 3143


## 3. Write the cleaned CSV with `csv.DictWriter`

In [ ]:
with CLEAN_PATH.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["fips", "name", "state"])
    writer.writeheader()
    writer.writerows(clean_rows)

print("Wrote", len(clean_rows), "rows ->", CLEAN_PATH.resolve())

Wrote 3143 rows -> /mnt/user-data/outputs/homework/week04/data/us_county_fips_clean.csv


## 4. Verify the result is join-ready

In [ ]:
with CLEAN_PATH.open(newline="", encoding="utf-8") as f:
    check = list(csv.DictReader(f))

all_5digit = all(len(r["fips"]) == 5 and r["fips"].isdigit() for r in check)
unique_key = len({r["fips"] for r in check}) == len(check)

print("rows                 :", len(check))
print("every FIPS is 5-digit:", all_5digit)          # -> matches polygon GEOID width
print("FIPS is a unique key :", unique_key)           # -> safe one-to-one join
print("Doña Ana repaired    :", next(r['name'] for r in check if r['fips']=='35013'))
print()
print("Before -> After (first county):")
print("  raw  :", {k: raw_rows[3][k] for k in ('fips','name','state')})   # Autauga, was '1001'
print("  clean:", check[0])

rows                 : 3143
every FIPS is 5-digit: True
FIPS is a unique key : True
Doña Ana repaired    : Doña Ana County

Before -> After (first county):
  raw  : {'fips': '1003', 'name': 'Baldwin County', 'state': 'AL'}
  clean: {'fips': '01001', 'name': 'Autauga County', 'state': 'AL'}


## 5. Using it for the join
The cleaned `fips` column now matches the `GEOID` attribute of a county vector layer exactly (both are 5-character strings), so a downstream join is a one-liner, e.g. with GeoPandas:

```python
counties = gpd.read_file("cb_2023_us_county_500k.shp")          # vector polygons
attrs    = pd.read_csv("data/us_county_fips_clean.csv", dtype={"fips": str})
joined   = counties.merge(attrs, left_on="GEOID", right_on="fips")   # 3,143 clean matches
```

**Result:** 3,143 county rows, every FIPS a zero-padded 5-digit string, aggregates removed, encoding repaired — ready to attach to county polygons with no silent match failures.